# SmartClassrooms Roles, explicado de principio a fin

Este cuaderno explica **cómo funciona todo el proyecto**, pieza por pieza, con ejemplos que se ejecutan de verdad sobre una foto de prueba.

**Idea del proyecto en una frase:** subes la foto de un salón, una IA de visión cuenta cuántas personas hay y qué rol tiene cada una (estudiante, aseo o seguridad), y después le preguntas a un **agente** en español ("¿qué tan lleno está el 38-101?"). El agente usa herramientas para averiguarlo y te responde.

| Parte | Qué verás |
|---|---|
| 1. Arquitectura | Las piezas del sistema y qué tecnología usa cada una |
| 2. ¿Qué es una imagen para un computador? | Píxeles, números y canales de color |
| 3. Convoluciones | El "filtro" que usan las redes neuronales para ver (con animación) |
| 4. Detección con YOLO | Cómo encuentra a las personas: cajas, confianza, IoU y NMS |
| 5. Clasificación con EfficientNetB0 | Cómo decide el rol de cada persona: capas, softmax y umbral |
| 6. Entrenamiento | Transfer learning, fine-tuning, aumentos, partición y métricas |
| 7. El agente | Qué es un LLM, cómo usa herramientas y el ciclo del agente (con animación) |
| 8. Todo junto | De la foto a la respuesta |
| 9. Limitaciones | Qué falla y por qué |

> **Fotos:** se usan fotos de ejemplo y dos fotos reales de salones de EAFIT, con el consentimiento de las personas.\n>\n> **Cómo ejecutarlo:** abre este archivo con el kernel del entorno `vision-service/.venv` (en VS Code o Jupyter) y ejecuta las celdas en orden. La primera vez descarga `yolov8m.pt`. No necesita el dataset ni la clave del LLM: la parte del agente tiene una demo simulada, y la llamada real solo se ejecuta si existe `NVIDIA_API_KEY`.

In [ ]:
# Preparación: librerías, modelos y la foto de ejemplo.
import json, os, hashlib, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # menos mensajes técnicos de TensorFlow
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import animation
from IPython.display import HTML, display
from PIL import Image

import roles_salon as rs   # el pipeline de visión del proyecto
import agente              # el agente con herramientas

plt.rcParams.update({"figure.dpi": 90, "axes.titlesize": 11, "font.size": 10})

# IMPORTANTE: YOLO (PyTorch) se carga ANTES que TensorFlow. En Linux, si TensorFlow
# arranca primero, las dos librerías chocan y el programa se cae (ver rs.get_yolo).
yolo = rs.get_yolo("yolov8m.pt")
rs.configurar_tf()
import keras

P = rs.rutas("data_roles")
clasificador = keras.saving.load_model(P["modelo"])
clases = json.loads(P["clases"].read_text(encoding="utf-8"))

FOTO = Path("fotos_prueba/salon_aseo.webp")
img = rs.cargar_imagen(FOTO)
print("Clases del clasificador:", clases)
print("Tamaño de la foto:", img.size)
plt.figure(figsize=(8, 5)); plt.imshow(img); plt.axis("off"); plt.title("Foto de ejemplo"); plt.show()

## 1. Arquitectura: las piezas del sistema

```
 Usuario ──► Página web (React) ──► API (FastAPI) ──┬──► Visión: YOLOv8m ──► EfficientNetB0 ──► conteo por rol
                                                     └──► Agente ──► LLM (NVIDIA NIM) ◄──► herramientas
```

| Pieza | Tecnología | Para qué sirve | Cómo funciona por debajo |
|---|---|---|---|
| Página web | **React + Vite + Tailwind** | Subir fotos, ver los recuadros y chatear con el agente | React arma la interfaz con componentes que se redibujan cuando cambia el estado. Vite sirve la página en desarrollo y reenvía `/analizar` y `/preguntar` a la API. Tailwind da estilos con clases (`rounded-xl`, `bg-blue-600`…) |
| API | **FastAPI + Uvicorn** | Recibe la foto o la pregunta y devuelve JSON | FastAPI convierte funciones de Python en direcciones web (`POST /analizar`) y valida los datos con Pydantic. Uvicorn es el servidor que atiende las peticiones |
| Lectura de imágenes | **Pillow** | Abrir JPG, PNG y WebP, rotarlas según el EXIF y recortar | Decodifica el archivo comprimido a una matriz de píxeles |
| Detector | **YOLOv8m** (Ultralytics, sobre **PyTorch**) | Encontrar a cada persona y dar su caja | Red convolucional preentrenada en COCO (80 tipos de objetos). Solo usamos la clase `person` |
| Clasificador | **EfficientNetB0** (**Keras** sobre **TensorFlow**) | Decidir el rol de cada persona recortada | Red convolucional preentrenada en ImageNet, a la que le enseñamos 3 clases nuevas (transfer learning) |
| Agente | **SDK de OpenAI** + **NVIDIA NIM** | Entender la pregunta, elegir herramientas y redactar la respuesta | Un modelo de lenguaje grande (LLM) que, en vez de inventar datos, pide ejecutar funciones de Python |
| Configuración | **python-dotenv** | Leer la clave del LLM desde `.env` sin escribirla en el código | Carga el archivo `.env` como variables de entorno |
| Métricas | **scikit-learn, NumPy, Matplotlib** | Matriz de confusión, F1 y gráficas | Cálculos sobre arreglos de números |

El dibujo de abajo muestra el mismo flujo con colores: **azul** es la interfaz, **verde** la visión y **morado** el agente.

In [ ]:
def caja(ax, x, y, w, h, texto, color):
    ax.add_patch(patches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                                        fc=color, ec="#333", lw=1))
    ax.text(x + w / 2, y + h / 2, texto, ha="center", va="center", fontsize=9)

def flecha(ax, a, b, texto=""):
    ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="->", lw=1.3, color="#444"))
    if texto:
        ax.text((a[0] + b[0]) / 2, (a[1] + b[1]) / 2 + 0.12, texto, ha="center", fontsize=8, color="#444")

fig, ax = plt.subplots(figsize=(12, 4.6)); ax.set_xlim(0, 12); ax.set_ylim(0, 5); ax.axis("off")
AZ, VE, MO, GR = "#dbeafe", "#dcfce7", "#ede9fe", "#f3f4f6"
caja(ax, 0.2, 2.0, 1.6, 1.0, "Usuario", GR)
caja(ax, 2.4, 2.0, 1.9, 1.0, "Página web\n(React)", AZ)
caja(ax, 4.9, 2.0, 1.9, 1.0, "API\n(FastAPI)", AZ)
caja(ax, 7.6, 3.4, 1.9, 1.0, "YOLOv8m\n(detecta)", VE)
caja(ax, 9.9, 3.4, 1.9, 1.0, "EfficientNetB0\n(clasifica)", VE)
caja(ax, 7.6, 0.6, 1.9, 1.0, "Agente\n(ciclo)", MO)
caja(ax, 9.9, 0.6, 1.9, 1.0, "LLM\n(NVIDIA NIM)", MO)
flecha(ax, (1.8, 2.5), (2.4, 2.5))
flecha(ax, (4.3, 2.5), (4.9, 2.5), "foto / pregunta")
flecha(ax, (6.8, 2.8), (7.6, 3.8), "/analizar")
flecha(ax, (9.5, 3.9), (9.9, 3.9), "recortes")
flecha(ax, (6.8, 2.2), (7.6, 1.2), "/preguntar")
flecha(ax, (9.5, 1.25), (9.9, 1.25)); flecha(ax, (9.9, 0.95), (9.5, 0.95))
ax.annotate("", xy=(8.55, 1.6), xytext=(8.55, 3.4), arrowprops=dict(arrowstyle="->", lw=1.3, ls="--", color="#16a34a"))
ax.text(8.65, 2.5, "conteo por rol\n(herramienta)", fontsize=8, color="#16a34a")
ax.set_title("Arquitectura de SmartClassrooms Roles"); plt.show()

## 2. ¿Qué es una imagen para un computador?

Para el computador, una foto es **una tabla de números**. Cada punto (píxel) tiene 3 valores entre 0 y 255: cuánto **rojo**, **verde** y **azul** tiene (RGB). Una foto de 1000×700 píxeles es una matriz de `700 × 1000 × 3` números.

Toda la IA de visión trabaja sobre esos números: no "ve" una persona, ve patrones en la tabla.

In [ ]:
arr = np.array(img)
print("Forma de la matriz:", arr.shape, "→ (alto, ancho, canales RGB)")
print("Un píxel cualquiera:", arr[300, 400], "= [rojo, verde, azul]")

fig, axs = plt.subplots(1, 4, figsize=(14, 3.4))
axs[0].imshow(arr); axs[0].set_title("Foto original")
for i, (nombre, cmap) in enumerate([("Rojo", "Reds"), ("Verde", "Greens"), ("Azul", "Blues")], 1):
    axs[i].imshow(arr[:, :, i - 1], cmap=cmap); axs[i].set_title(f"Canal {nombre}")
for a in axs: a.axis("off")
plt.show()

# Un pedacito de 8×8 píxeles con sus números (canal verde)
y0, x0 = 300, 400
parche = arr[y0:y0 + 8, x0:x0 + 8, 1]
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.imshow(parche, cmap="gray", vmin=0, vmax=255)
for (i, j), v in np.ndenumerate(parche):
    ax.text(j, i, v, ha="center", va="center", fontsize=8, color="red")
ax.set_title("8×8 píxeles = 64 números (canal verde)"); ax.axis("off"); plt.show()

## 3. Convoluciones: cómo "mira" una red neuronal

Las redes que usamos (YOLO y EfficientNet) son **redes neuronales convolucionales (CNN)**. Su operación básica es la **convolución**:

1. Se toma un **filtro** (kernel), una tablita pequeña de números, por ejemplo de 3×3.
2. Se **desliza** sobre la imagen. En cada posición se multiplica el filtro por los píxeles que tapa y se **suman** los resultados.
3. El número que sale indica **cuánto se parece esa zona al patrón del filtro**.

Un filtro como `[[-1,0,1],[-1,0,1],[-1,0,1]]` responde fuerte en los **bordes verticales**. La red **aprende sola** miles de filtros durante el entrenamiento:
- Las primeras capas detectan bordes y colores.
- Las capas intermedias detectan texturas, como la tela de un uniforme.
- Las últimas detectan partes completas: caras, cuerpos o una escoba.

In [ ]:
def convolucionar(gris, kernel):
    k = kernel.shape[0]
    ventanas = np.lib.stride_tricks.sliding_window_view(gris, (k, k))
    return (ventanas * kernel).sum(axis=(-1, -2))

gris = np.array(img.convert("L").resize((480, 320)), dtype=float)
filtros = {
    "Bordes verticales": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    "Bordes horizontales": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
    "Desenfoque": np.ones((5, 5)) / 25,
}
fig, axs = plt.subplots(1, 4, figsize=(15, 3.2))
axs[0].imshow(gris, cmap="gray"); axs[0].set_title("Original en gris")
for a, (nombre, k) in zip(axs[1:], filtros.items()):
    a.imshow(np.abs(convolucionar(gris, k)), cmap="gray"); a.set_title(nombre)
for a in axs: a.axis("off")
plt.show()

### Animación: el filtro deslizándose

El cuadro rojo (3×3) recorre la imagen pequeña de la izquierda. En cada paso, la suma de "píxeles × filtro" llena una casilla del resultado de la derecha. Este filtro detecta un borde vertical: en el resultado se ve el borde iluminado.

In [ ]:
mini = np.zeros((8, 8)); mini[:, 4:] = 1          # imagen con un borde vertical en el medio
k = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
salida = convolucionar(mini, k)                    # 6×6
parcial = np.full_like(salida, np.nan)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(8, 4))
a1.imshow(mini, cmap="gray", vmin=0, vmax=1); a1.set_title("Imagen (8×8)")
marco = patches.Rectangle((-0.5, -0.5), 3, 3, fill=False, ec="red", lw=3); a1.add_patch(marco)
im2 = a2.imshow(parcial, cmap="viridis", vmin=salida.min(), vmax=salida.max()); a2.set_title("Resultado (6×6)")
texto = a2.text(0, -0.9, "", fontsize=10)
for a in (a1, a2): a.set_xticks([]); a.set_yticks([])

def paso(n):
    i, j = divmod(n, salida.shape[1])
    marco.set_xy((j - 0.5, i - 0.5))
    parcial[i, j] = salida[i, j]; im2.set_data(parcial)
    texto.set_text(f"posición ({i},{j}) → suma = {salida[i, j]:.0f}")
    return marco, im2, texto

anim = animation.FuncAnimation(fig, paso, frames=salida.size, interval=250, blit=False)
plt.close(fig)
display(HTML(anim.to_jshtml()))

## 4. Detección de personas con YOLOv8m

**YOLO** (*You Only Look Once*) mira la foto **una sola vez** y devuelve, para cada objeto que encuentra:
- una **caja** (x1, y1, x2, y2) en píxeles,
- una **clase** (en COCO, la 0 es `person`),
- una **confianza** entre 0 y 1.

**Cómo funciona por dentro:**
1. Una **columna vertebral** (*backbone*) de convoluciones convierte la foto en mapas de características.
2. Un **cuello** (*neck*) mezcla esos mapas a varias escalas, para ver tanto personas grandes como pequeñas.
3. Una **cabeza** (*head*) predice, para cada punto de una cuadrícula, si ahí hay un objeto, de qué clase y dónde está su caja.

Esto produce **miles de cajas candidatas**, muchas repetidas sobre la misma persona. Por eso al final se aplica **NMS** (*Non-Maximum Suppression*, explicado abajo).

**Decisiones del proyecto:**
- **Tamaño de entrada de 960 px.** A 640 px se perdían los estudiantes del fondo; a 1280 px aparecían personas que no existen.
- **Confianza mínima de 0,40.**

In [ ]:
res = yolo.predict(img, conf=0.40, imgsz=960, classes=[0], verbose=False)[0]
cajas = res.boxes.xyxy.cpu().numpy(); confs = res.boxes.conf.cpu().numpy()
print(f"YOLO encontró {len(cajas)} personas")

fig, ax = plt.subplots(figsize=(10, 6.5)); ax.imshow(img); ax.axis("off")
for (x1, y1, x2, y2), c in zip(cajas, confs):
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="lime", lw=2))
    ax.text(x1, y1 - 3, f"{c:.0%}", color="black", fontsize=7, bbox=dict(fc="lime", ec="none", pad=1))
ax.set_title("Detecciones de YOLOv8m (persona + confianza)"); plt.show()

### IoU y NMS: cómo se eliminan las cajas repetidas

- **IoU** (*Intersection over Union*) mide cuánto se superponen dos cajas: el área donde se cruzan dividida entre el área total que cubren. Vale 0 si no se tocan y 1 si son idénticas.
- **NMS:**
  1. Ordena las cajas por confianza.
  2. Se queda con la mejor y **borra** las que se superponen mucho con ella (IoU > 0,7).
  3. Repite con las que quedan.

In [ ]:
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter)

def nms(cajas, confs, umbral=0.7):
    orden, quedan = list(np.argsort(-confs)), []
    while orden:
        mejor = orden.pop(0); quedan.append(mejor)
        orden = [i for i in orden if iou(cajas[mejor], cajas[i]) < umbral]
    return quedan

# Ejemplo inventado: 3 cajas sobre la misma persona y 1 sobre otra
cand = np.array([[10, 10, 50, 90], [12, 14, 52, 92], [8, 6, 47, 86], [70, 20, 100, 80]])
cconf = np.array([0.92, 0.85, 0.60, 0.88])
quedan = nms(cand, cconf)
print("IoU entre la caja 0 y la 1:", round(iou(cand[0], cand[1]), 2))
print("Cajas que sobreviven a NMS:", [int(i) for i in quedan], "(las 3 repetidas quedan en 1)")

fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))
for a, titulo, idx in [(axs[0], "Antes de NMS (4 cajas)", range(4)), (axs[1], "Después de NMS", quedan)]:
    a.set_xlim(0, 110); a.set_ylim(100, 0); a.set_title(titulo); a.set_xticks([]); a.set_yticks([])
    for i in idx:
        x1, y1, x2, y2 = cand[i]
        a.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=f"C{i}", lw=2))
        a.text(x1, y1 - 2, f"{cconf[i]:.2f}", color=f"C{i}", fontsize=8)
plt.show()

In [ ]:
# ¿Por qué 960 px? Mismo modelo, misma foto, distinto tamaño de entrada:
for tam in (640, 960, 1280):
    n = len(yolo.predict(img, conf=0.40, imgsz=tam, classes=[0], verbose=False)[0].boxes)
    print(f"imgsz={tam:5d} → {n} personas")

## 5. Clasificación del rol con EfficientNetB0

YOLO sabe que hay una **persona**, pero no si es estudiante, de aseo o de seguridad. Para eso:
1. Se **recorta** cada persona con un 8 % de margen.
2. El recorte se redimensiona a **224×224**.
3. Pasa por **EfficientNetB0**, una CNN preentrenada en ImageNet (1,2 millones de fotos y 1.000 clases).

El modelo tiene esta forma:

```
recorte 224×224×3
  → aumentos (solo al entrenar: volteos, giros, brillo)
  → EfficientNetB0 (238 capas, convierte la imagen en 1.280 números que la describen)
  → GlobalAveragePooling (promedia cada mapa en un número)
  → Dropout 0,3 (apaga neuronas al azar al entrenar, para no memorizar)
  → Dense de 3 neuronas + softmax  →  [P(estudiante), P(aseo), P(seguridad)]
```

In [ ]:
clasificador.summary(line_length=90, expand_nested=False)

### Lo que "ve" la primera capa

Estos son algunos de los 32 **mapas de características** que produce la primera convolución de EfficientNet para un recorte. Cada mapa es la respuesta a un filtro aprendido: unos se iluminan con los bordes, otros con colores o texturas.

In [ ]:
recortes = [rs.recortar_con_margen(img, b) for b in cajas]
r0 = recortes[int(np.argmax([(b[2] - b[0]) * (b[3] - b[1]) for b in cajas]))]   # la persona más grande
x0 = np.array(r0.resize(rs.IMG_SIZE), dtype="float32")[None]

base = clasificador.get_layer("efficientnetb0")
primera = keras.Model(base.inputs, base.get_layer("stem_activation").output)
mapas = primera(x0).numpy()[0]          # 112×112×32

fig, axs = plt.subplots(2, 6, figsize=(14, 5))
axs[0, 0].imshow(r0); axs[0, 0].set_title("Recorte")
for a, c in zip(axs.flat[1:], range(11)):
    a.imshow(mapas[:, :, c], cmap="magma"); a.set_title(f"filtro {c}")
for a in axs.flat: a.axis("off")
plt.suptitle("Mapas de características de la primera capa"); plt.show()

### Softmax: de números sueltos a probabilidades

La última capa da 3 números cualesquiera, llamados *logits*. La función **softmax** los convierte en probabilidades que suman 1:

$$P(\text{clase}_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Ejemplo: los logits `[2.0, 0.5, -1.0]` se convierten en las probabilidades `[0.79, 0.18, 0.04]`.

**Umbral de "indeterminado":** si la probabilidad más alta es menor de **0,60**, el proyecto no adivina. Cuenta a la persona como "indeterminada": suma al total, pero no a ningún rol.

In [ ]:
z = np.array([2.0, 0.5, -1.0]); print("softmax:", np.round(np.exp(z) / np.exp(z).sum(), 2))

X = np.stack([np.array(r.resize(rs.IMG_SIZE), dtype="float32") for r in recortes])
probs = clasificador.predict(X, verbose=0)

# Primero las personas que NO son estudiantes, para que se vea más de un rol.
orden = sorted(range(len(recortes)), key=lambda i: (probs[i].argmax() == 0, -probs[i].max()))[:8]
n = len(orden)
fig, axs = plt.subplots(2, n, figsize=(2.1 * n, 5), gridspec_kw={"height_ratios": [3, 1.3]})
for i, idx in enumerate(orden):
    p = probs[idx]; k = int(p.argmax())
    rol = clases[k] if p[k] >= 0.60 else "indeterminado"
    axs[0, i].imshow(recortes[idx]); axs[0, i].axis("off")
    axs[0, i].set_title(f"{rs.ROLE_LABELS[rol]}\n{p[k]:.0%}", fontsize=9, color=rs.ROLE_COLORS[rol])
    axs[1, i].barh([rs.ROLE_LABELS[c] for c in clases], p, color=[rs.ROLE_COLORS[c] for c in clases])
    axs[1, i].set_xlim(0, 1); axs[1, i].axvline(0.6, ls="--", c="gray", lw=0.8)
    axs[1, i].tick_params(labelsize=7)
    if i: axs[1, i].set_yticks([])
plt.suptitle("Probabilidades del clasificador por persona (línea gris = umbral 0,60)"); plt.tight_layout(); plt.show()

In [ ]:
# El resultado final del pipeline de visión (lo que usa la API y el agente):
_, detecciones, conteo = rs.analizar_salon(FOTO, yolo, clasificador, clases)
print("Conteo:", dict(conteo))
plt.figure(figsize=(10, 6.5)); plt.imshow(rs.dibujar_resultados(img, detecciones, conteo)); plt.axis("off"); plt.show()

## 6. Cómo se entrenó el clasificador

### 6.1 Transfer learning, o por qué no empezamos de cero

Entrenar una CNN desde cero necesita millones de fotos. Nosotros teníamos unas 750. La solución es el **transfer learning**: reutilizar EfficientNetB0, que ya sabe "ver" (bordes, texturas, formas, personas), y enseñarle solo lo nuevo. Se hace en dos fases:

| Fase | Qué se entrena | Velocidad de aprendizaje | Analogía |
|---|---|---|---|
| 1. Base congelada | Solo la capa final de 3 neuronas | 0,001 | Contratas a un experto en ropa y solo le explicas qué uniforme es de qué rol |
| 2. Fine-tuning | Además, las últimas 20 capas de EfficientNet | 0,00001 (100 veces menor) | Le dejas reaprender a fijarse en detalles: chalecos, escobas, placas |

En la fase 2 la velocidad es tan baja para no "olvidar" lo que la red ya sabía. Las capas de BatchNorm se dejan congeladas porque, con lotes pequeños, sus estadísticas se vuelven inestables.

### 6.2 El ciclo de aprendizaje

En cada paquete de 32 fotos de *train*:
1. **Predice** las probabilidades.
2. **Mide el error** con la *entropía cruzada*, el `loss` que aparece en pantalla.
3. **Retropropagación** (*backpropagation*): calcula cómo mover cada peso para bajar el error.
4. **Adam** mueve los pesos un poquito en esa dirección.

Una **época** es una vuelta completa a todas las fotos. **Early stopping** detiene el entrenamiento si `val_loss` no mejora en 3 épocas, y se queda con la mejor época.

In [ ]:
# Aumentos de datos: la misma foto se ve un poco distinta en cada época, para que el modelo
# no memorice. No se cambia el tono de color porque el color del uniforme es una pista útil.
aug = clasificador.get_layer("augmentation")
fig, axs = plt.subplots(1, 6, figsize=(13, 3))
axs[0].imshow(x0[0].astype("uint8")); axs[0].set_title("Original")
for a in axs[1:]:
    a.imshow(np.clip(aug(x0, training=True).numpy()[0], 0, 255).astype("uint8")); a.set_title("Aumentada")
for a in axs: a.axis("off")
plt.show()

### 6.3 Partición de los datos: train, validation y test

| Grupo | % | Para qué | Analogía |
|---|---|---|---|
| train | 70 % | Aprender | Los ejercicios que estudias |
| validation | 15 % | Decidir cuándo parar y qué versión guardar | Los simulacros |
| test | 15 % | La nota final, con fotos que nunca vio | El examen |

Reglas del proyecto:
- **Por foto de origen:** los recortes de una misma foto van al mismo grupo. Si no, el modelo "vería el examen" mientras estudia.
- **Partición fija:** las fotos se ordenan por un **hash** de su nombre (una huella que siempre da lo mismo), y los grupos se llenan en ese orden hasta tener el 15/15/70 % de los recortes. Así el examen no cambia entre entrenamientos y las notas se pueden comparar.

In [ ]:
fotos = {f"{i:04d}": 1 + i % 3 for i in range(20)}      # 20 fotos inventadas con 1 a 3 recortes
reparto = rs.particion_fija("estudiante", fotos)
for f in list(fotos)[:6]:
    h = hashlib.sha256(f"estudiante/{f}".encode()).hexdigest()[:10]
    print(f"foto {f}  hash {h}…  recortes={fotos[f]}  → {reparto[f]}")
total = sum(fotos.values())
print("Con solo 20 fotos los porcentajes son aproximados; con cientos quedan en 70/15/15.")
for g in ("train", "validation", "test"):
    n = sum(v for f, v in fotos.items() if reparto[f] == g); print(f"{g:10s} {n:2d} recortes ({n / total:.0%})")

### 6.4 Resultados y métricas

- **Accuracy** (exactitud): aciertos entre el total.
- **Precisión:** de las veces que dijo "seguridad", ¿cuántas acertó?
- **Recall:** de los guardias reales, ¿cuántos encontró?
- **F1:** combina precisión y recall. El **F1 macro** promedia las 3 clases con el mismo peso, para que la clase más grande no maquille la nota.
- **Intervalo de confianza (bootstrap):** se remuestrea el test 2.000 veces para ver cuánto podría variar la nota. Con un test pequeño, el margen es de unos ±5 puntos.

Resultado final en test (111 recortes): **accuracy 0,910 y F1 macro 0,911**. La base congelada sola daba 0,901.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))
for a, nombre in zip(axs, ["curvas_fine_tuning.png", "confusion_antes.png", "confusion_despues.png"]):
    a.imshow(plt.imread(P["modelo_dir"] / nombre)); a.axis("off")
plt.show()

# Bootstrap con un ejemplo inventado de 100 respuestas (91 aciertos):
rng = np.random.default_rng(0)
y_true = rng.integers(0, 3, 100); y_pred = y_true.copy(); y_pred[:9] = (y_pred[:9] + 1) % 3
lo, hi = rs.intervalo_bootstrap(y_true, y_pred)
print(f"accuracy = {np.mean(y_true == y_pred):.2f}, IC 95 % ≈ {lo:.2f}–{hi:.2f}")

## 7. El agente con herramientas

### 7.1 ¿Qué es un LLM?

Un **LLM** (*Large Language Model*), como Nemotron, Llama o GPT, es una red neuronal enorme (de tipo **Transformer**) entrenada para **predecir la siguiente palabra** de un texto. Para eso divide el texto en *tokens* (pedazos de palabras), y usa **atención** para decidir qué partes del texto anterior importan para la siguiente palabra. Con suficiente entrenamiento aprende a conversar, razonar y seguir instrucciones.

Su problema es que **no sabe nada de tus salones**: si le preguntas cuántas personas hay en el 38-101, inventaría. Por eso no le pedimos los datos: le damos **herramientas**.

### 7.2 Tool calling (llamado de herramientas)

1. Le describimos al LLM cada herramienta con un **esquema JSON**: nombre, qué hace y qué argumentos recibe.
2. El LLM no ejecuta nada. Si necesita un dato, responde con un **pedido estructurado**, por ejemplo `contar_personas({"salon": "38-101"})`.
3. **Nuestro código** ejecuta la función de Python y le devuelve el resultado como un mensaje `tool`.
4. El LLM lee el resultado y decide si pide otra herramienta o si ya responde.

Así el LLM pone **el lenguaje y el razonamiento**, y las herramientas ponen **los datos y los cálculos exactos**.

| Herramienta | Argumentos | Devuelve |
|---|---|---|
| `listar_salones` | ninguno | Salones, capacidad y si tienen foto analizada |
| `contar_personas` | `salon` | Total y conteo por rol de la última foto |
| `calcular_ocupacion` | `salon` | Personas, capacidad, porcentaje, cupos libres y si se pasa de la capacidad. Toma los números de los datos, no del LLM |

In [ ]:
# Este es el esquema que recibe el LLM para una de las herramientas:
print(json.dumps(agente.ESQUEMAS[1], ensure_ascii=False, indent=2))
print("\nInstrucciones del sistema (lo primero que lee el LLM):\n")
print(agente.SISTEMA)

### 7.3 El ciclo del agente (`agente.preguntar`)

```
mensajes = [sistema, pregunta]
repetir:
    respuesta = LLM(mensajes, herramientas)
    si no pide herramientas → devolver su texto  (fin)
    si pide herramientas    → ejecutarlas y agregar sus resultados a mensajes
    si ya hubo 6 llamadas   → quitarle las herramientas: debe responder con lo que tiene
```

El **límite de 6 llamadas** evita que el agente entre en un bucle infinito, consultando una y otra vez y gastando créditos. Los **errores** de una herramienta, como un salón que no existe, no detienen el programa: vuelven al LLM como texto, para que los explique.

La animación muestra el recorrido de la pregunta *"¿Qué tan lleno está el 38-101?"*.

In [ ]:
pasos_anim = [
    ("Usuario", "¿Qué tan lleno está el 38-101?"),
    ("LLM", "Necesito la ocupación → pide calcular_ocupacion('38-101')"),
    ("Herramienta", "{'personas': 16, 'capacidad': 40, 'ocupacion_pct': 40.0, 'cupos_libres': 24}"),
    ("LLM", "Ya tengo todo → responde"),
    ("Usuario", "«El 38-101 está al 40 %: 16 de 40, quedan 24 cupos.»"),
]
pos = {"Usuario": (1.2, 2.0), "LLM": (5.0, 2.0), "Herramienta": (8.8, 2.0)}
col = {"Usuario": "#dbeafe", "LLM": "#ede9fe", "Herramienta": "#dcfce7"}

fig, ax = plt.subplots(figsize=(10, 3.8)); ax.set_xlim(0, 10); ax.set_ylim(0, 4); ax.axis("off")
cajas_anim = {}
for nombre, (x, y) in pos.items():
    r = patches.FancyBboxPatch((x - 0.9, y - 0.45), 1.8, 0.9, boxstyle="round,pad=0.05", fc=col[nombre], ec="#999", lw=1)
    ax.add_patch(r); cajas_anim[nombre] = r
    ax.text(x, y, nombre, ha="center", va="center", fontsize=10, weight="bold")
mensaje = ax.text(5, 3.4, "", ha="center", fontsize=10, wrap=True)
contador = ax.text(5, 0.6, "", ha="center", fontsize=9, color="#555")
linea = ax.annotate("", xy=(0, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="->", lw=2, color="#e11d48"))

def cuadro(n):
    actor, texto = pasos_anim[n]
    for nombre, r in cajas_anim.items():
        r.set_linewidth(3 if nombre == actor else 1); r.set_edgecolor("#e11d48" if nombre == actor else "#999")
    previo = pasos_anim[n - 1][0] if n else actor
    linea.xy, linea.xyann = pos[actor], pos[previo]
    mensaje.set_text(texto)
    llamadas = sum(1 for a, _ in pasos_anim[: n + 1] if a == "Herramienta")
    contador.set_text(f"paso {n + 1}/{len(pasos_anim)} · llamadas a herramientas: {llamadas} (límite 6)")
    return []

anim = animation.FuncAnimation(fig, cuadro, frames=len(pasos_anim), interval=1300)
plt.close(fig)
display(HTML(anim.to_jshtml()))

### 7.4 Demo con un LLM simulado (no necesita clave)

Para ver el ciclo **real** de `agente.py` sin gastar créditos, reemplazamos el LLM por uno "de mentira" que responde un guion fijo. Todo lo demás es el código del proyecto: la ejecución de herramientas, los mensajes `tool` y el límite. Se imprime cada mensaje que viaja.

In [ ]:
from types import SimpleNamespace as NS

def pedido(nombre, **args):
    return NS(id=f"id_{nombre}", function=NS(name=nombre, arguments=json.dumps(args)))

class LLMSimulado:
    def __init__(self, guion):
        self.guion = list(guion); self.chat = NS(completions=NS(create=self.create))
    def create(self, messages, **kw):
        print(f"\n── El LLM recibe {len(messages)} mensajes. El último es de '{messages[-1]['role']}':")
        print("   ", str(messages[-1].get("content"))[:150])
        t = self.guion.pop(0)
        msg = NS(content=t, tool_calls=None) if isinstance(t, str) else NS(content="", tool_calls=t)
        print("── El LLM contesta:", t if isinstance(t, str) else [f"{x.function.name}({x.function.arguments})" for x in t])
        return NS(choices=[NS(message=msg)])

agente.ANALISIS.clear()
agente.registrar_analisis("38-101", len(detecciones), {rs.ROLE_LABELS[r]: n for r, n in conteo.items()})

llm = LLMSimulado([
    [pedido("contar_personas", salon="38-101")],
    [pedido("calcular_ocupacion", salon="38-101")],
    f"El 38-101 tiene {len(detecciones)} personas: {len(detecciones) / 40:.0%} de su capacidad.",
])
r = agente.preguntar("¿Qué tan lleno está el 38-101?", llm)
print("\n════ RESULTADO ════")
for i, p in enumerate(r["pasos"], 1):
    print(f"{i}. {p['herramienta']}({p['argumentos']}) → {p['resultado']}")
print("Respuesta:", r["respuesta"])

### 7.5 Llamada real al LLM (solo si tienes la clave)

Si existe `NVIDIA_API_KEY` (en el entorno o en `vision-service/.env`), esta celda le pregunta al modelo real `nvidia/nemotron-3.5-super-vl-120b-a12b`. Si no existe, solo avisa.

In [ ]:
try:
    cliente = agente.crear_cliente()
except RuntimeError as e:
    print("Sin clave, se omite la llamada real:", e)
else:
    for q in ["¿Hay personal de aseo en el 38-101?", "¿Cuántos guardias hay en el 38-201?"]:
        r = agente.preguntar(q, cliente)
        print(f"\nPregunta: {q}")
        for p in r["pasos"]:
            print(f"   → {p['herramienta']}({p['argumentos']})")
        print("Respuesta:", r["respuesta"])

## 8. Todo junto: de la foto a la respuesta

```
1. En la pestaña Fotos, el usuario sube la foto del 38-101 y pulsa Procesar
2. La página envía POST /analizar con la foto y el salón
3. La API valida la imagen (≤ 15 MB, formato válido)
4. YOLOv8m (960 px) detecta las personas
5. Cada persona se recorta y EfficientNetB0 decide su rol (umbral 0,60)
6. La API guarda SOLO el conteo del salón (la foto se descarta) y devuelve las cajas
7. En la pestaña Asistente, el usuario pregunta "¿qué tan lleno está el 38-101?"
8. La página envía POST /preguntar
9. El agente arranca el ciclo: el LLM pide herramientas → se ejecutan → el LLM lee los resultados
10. La respuesta y los pasos (herramienta, argumentos y resultado) vuelven a la página
```

**Privacidad:** el LLM nunca ve la foto, solo números. Al pulsar "Quitar foto", también se borra el conteo (`DELETE /analisis/{salon}`).

### 8.1 Caso real: fotos de salones de EAFIT

Hasta ahora usamos una foto de ejemplo. Estas dos son **fotos reales de EAFIT**, que no están en el dataset de entrenamiento; las personas dieron su consentimiento para usarlas. Aquí se ve el sistema completo en condiciones reales, incluidos sus errores:
- **`eafit_estudiantes_de_pie.jpg`:** tres estudiantes exponiendo de pie frente al tablero. Es el caso que antes confundía al modelo con personal de aseo.
- **`eafit_evento.jpg`:** un salón con personas tapadas por otras y una expositora. Es el caso más difícil.

In [ ]:
reales = {"38-102": Path("fotos_prueba/eafit_estudiantes_de_pie.jpg"), "38-201": Path("fotos_prueba/eafit_evento.jpg")}
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
for a, (salon, ruta) in zip(axs, reales.items()):
    im, dets, cnt = rs.analizar_salon(ruta, yolo, clasificador, clases)
    agente.registrar_analisis(salon, len(dets), {rs.ROLE_LABELS[r]: n for r, n in cnt.items()})
    a.imshow(rs.dibujar_resultados(im, dets, cnt)); a.axis("off")
    a.set_title(f"{salon}: {ruta.name}")
    print(f"{salon} ({ruta.name}): {len(dets)} personas → {dict(cnt)}")
plt.tight_layout(); plt.show()

In [ ]:
# Ahora el agente responde sobre estos salones (con el LLM real si hay clave; si no, se omite).
try:
    cliente = agente.crear_cliente()
except RuntimeError as e:
    print("Sin clave, se omite:", e)
else:
    for q in ["¿Cuántas personas hay en el 38-201 y hay alguien de aseo o seguridad?",
              "¿Hay personal de aseo en el 38-102?",
              "Compara la ocupación del 38-101, el 38-102 y el 38-201."]:
        r = agente.preguntar(q, cliente)
        print(f"\nPregunta: {q}")
        for p in r["pasos"]:
            print(f"   → {p['herramienta']}({p['argumentos']})")
        print("Respuesta:", r["respuesta"])

**Qué observar:**
- En `eafit_evento.jpg`, YOLO no detecta a las personas casi tapadas, y la expositora queda contada como estudiante. El agente repite esos números con total seguridad, porque **no ve la foto**: solo recibe lo que devuelve la herramienta.
- En `eafit_estudiantes_de_pie.jpg`, revisa si los tres estudiantes de pie salen como estudiantes. Ese era el sesgo de "de pie y con ropa oscura = personal" que se corrigió agregando universitarios al dataset.
- La tercera pregunta compara 3 salones con 3 llamadas a `calcular_ocupacion`, una por salón. En la primera versión la herramienta recibía `personas` y `capacidad`: la misma pregunta necesitaba unas 7 llamadas y chocaba con el **límite de 6**. Pasar solo el salón resolvió las dos cosas: menos llamadas y ningún número puesto por el LLM.

## 9. Limitaciones y posibles mejoras

| Limitación | Por qué pasa | Posible mejora |
|---|---|---|
| Personas tapadas que no se cuentan | YOLO no ve a quien está casi oculto detrás de otra persona | Bajar la confianza solo para personas parcialmente visibles, o usar varias fotos |
| Profesores y expositores contados como estudiantes | El clasificador solo conoce 3 roles | Agregar una clase "profesor/expositor" |
| Estudiantes de pie con ropa oscura confundidos con personal | En los datos, aseo y seguridad casi siempre salen de pie y de uniforme oscuro (**sesgo del dataset**) | Más fotos de estudiantes de pie y con ropa oscura, que ya se agregaron en parte |
| El agente repite los errores de la visión | No ve la foto; confía en la herramienta | Que `contar_personas` informe cuántas detecciones tienen baja confianza |
| Test pequeño (≈ 35 recortes por clase) | Pocas fotos | Validación cruzada (k-fold) y más datos de salones reales |
| Datos de internet | Fotos de stock, no salones reales | Recolectar fotos de salones reales con permiso |

---
**Archivos clave:**
- `vision-service/roles_salon.py`: pipeline de visión y entrenamiento.
- `vision-service/agente.py`: herramientas y ciclo del agente.
- `vision-service/app.py`: la API.
- `frontend/src/app/App.tsx`: la página.
- `vision-service/casos_agente.md`: los 5 casos de prueba con el LLM real.